In this notebook we shows how to invoke *Ollama* (locally) and *API* (online) to produce structured outputs (in JSON schema).

### 1. Structured Output (JSON Format) Using Ollama + Locally-Deployed Models

See [Ollama's manual for developers](https://docs.ollama.com/capabilities/structured-outputs) for details on how to produce structured output. Ollama can be perceived as a container for downloaded LLMs, allowing users to run the models on their local terminals.
1. Download and install Ollama: https://ollama.com/download
2. Select from the list of LLMs available to download: https://ollama.com/library
3. Open command line interface (search "cmd" in Windows or "Terminal" on Mac) and run `ollama pull [NAME-OF-LLM]` (e.g., `ollama pull qwen3:30b-a3b`)
4. Restart Ollama and select the LLM just downloaded below the text bar. Then enter http://localhost:11434/ in the URL bar of your Internet Explorer. If it shows "Ollama is running", then you may run the code below.

In [ ]:
from ollama import chat
from pydantic import BaseModel

class Country(BaseModel): # Define a class of Objects to force LLM outputs into specific data types (string/integer/floats/lists...). 
  name: str
  capital: str
  languages: list[str]

response = chat(
  model='qwen3:30b-a3b', # Clarify the LLM to deploy (make sure the model is already downloaded through Ollama)
  # Impart the prompt to LLM. Two parameters are compulsory: (1)'role' (whose value must be 'system' or 'user'); (2)'content' (text of the prompt)
  messages=[{'role': 'user',
             'content': 'Tell me about Canada.'}],
  format=Country.model_json_schema(), # Constrain the output format to JSON schema
  think=False # Forbid invoking 'think' mode of the model (otherwise FAR more time-consuming)
)

output = Country.model_validate_json(response.message.content) # Extract structured output from the response returned by Ollama
print(output)

Country(name='Canada', capital='Ottawa', languages=['English', 'French', 'Indigenous languages', 'immigrant languages'])

### 2.1 Non-Structured Output (Plain Text) using QWEN API

Note: API Key is required and should be stored in a local environment variable named "DASHSCOPE_API_KEY".

In [ ]:
import os
from openai import OpenAI

try:
    client = OpenAI(
        # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为: api_key="sk-xxx",
        # IMPORTANT: API Key is stored as a local environment variable
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    completion = client.chat.completions.create(
        model="qwen3.6-flash",  # 模型列表: https://help.aliyun.com/model-studio/getting-started/models
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user', 'content': '你是谁？'}
        ]
    )
    print(completion.choices[0].message.content)
except Exception as e:
    print(f"错误信息：{e}")
    print("请参考文档：https://help.aliyun.com/model-studio/developer-reference/error-code")

你好！我是通义千问（Qwen），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。很高兴与你交流！有什么我可以帮你的吗？


### 2.2 Structured Output (JSON Format) Using QWEN API

In [ ]:
## See https://help.aliyun.com/zh/model-studio/qwen-structured-output

from openai import OpenAI
import os

client = OpenAI(
    # 如果没有配置环境变量，请用API Key将下行替换为：api_key="sk-xxx"
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 以下是北京地域base_url，如果使用新加坡地域的模型，需要将base_url替换为：https://dashscope-intl.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.6-flash",
    messages=[
        {
            "role": "system",
            "content": "请抽取用户的姓名与年龄信息，以JSON格式返回"
        },
        {
            "role": "user",
            "content": "大家好，我叫刘五，今年34岁，邮箱是liuwu@example.com，平时喜欢打篮球和旅游", 
        },
    ],
    response_format={"type": "json_object"}
)

json_string = completion.choices[0].message.content
print(json_string)

{
  "姓名": "刘五",
  "年龄": 34
}


### 2.3 Structured Output (JSON Format) Using QWEN API, with Examples

In [5]:
from openai import OpenAI
import os
import json
import textwrap  # 用于处理多行字符串的缩进，提高代码可读性

# 预定义示例响应，用于向模型展示期望的输出格式
# 示例1：包含所有字段的完整响应
example1_response = json.dumps(
    {
        "info": {"name": "张三", "age": "25岁", "email": "zhangsan@example.com"},
        "hobby": ["唱歌"]
    },
    ensure_ascii=False
)
# 示例2：包含多个hobby的响应
example2_response = json.dumps(
    {
        "info": {"name": "李四", "age": "30岁", "email": "lisi@example.com"},
        "hobby": ["跳舞", "游泳"]
    },
    ensure_ascii=False
)
# 示例3：不包含hobby字段的响应（hobby非必需）
example3_response = json.dumps(
    {
        "info": {"name": "赵六", "age": "28岁", "email": "zhaoliu@example.com"}
    },
    ensure_ascii=False
)
# 示例4：另一个不包含hobby字段的响应
example4_response = json.dumps(
    {
        "info": {"name": "孙七", "age": "35岁", "email": "sunqi@example.com"}
    },
    ensure_ascii=False
)

# 初始化OpenAI客户端
client = OpenAI(
    # 若没有配置环境变量，请将下行替换为：api_key="sk-xxx"
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 以下是北京地域base_url，如果使用新加坡地域的模型，需要将base_url替换为：https://dashscope-intl.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 使用dedent去除字符串缩进，使多行字符串在代码中美观，但运行时不包含额外空格
system_prompt = textwrap.dedent(f"""\
    请从用户输入中提取个人信息并按照指定的JSON Schema格式输出：

    【输出格式要求】
    输出必须严格遵循以下JSON结构：
    {{
      "info": {{
        "name": "字符串类型，必需字段，用户姓名",
        "age": "字符串类型，必需字段，格式为'数字+岁'，例如'25岁'",
        "email": "字符串类型，必需字段，标准邮箱格式，例如'user@example.com'"
      }},
      "hobby": ["字符串数组类型，非必需字段，包含用户的所有爱好，如未提及则完全不输出此字段"]
    }}

    【字段提取规则】
    1. name: 从文本中识别用户姓名，必需提取
    2. age: 识别年龄信息，转换为"数字+岁"格式，必需提取
    3. email: 识别邮箱地址，保持原始格式，必需提取
    4. hobby: 识别用户爱好，以字符串数组形式输出，如未提及爱好信息则完全省略hobby字段

    【参考示例】
    示例1（包含爱好）：
    Q：我叫张三，今年25岁，邮箱是zhangsan@example.com，爱好是唱歌
    A：{example1_response}

    示例2（包含多个爱好）：
    Q：我叫李四，今年30岁，邮箱是lisi@example.com，平时喜欢跳舞和游泳
    A：{example2_response}

    示例3（不包含爱好）：
    Q：我叫赵六，今年28岁，我的邮箱是zhaoliu@example.com
    A：{example3_response}

    示例4（不包含爱好）：
    Q：我是孙七，35岁，邮箱sunqi@example.com
    A：{example4_response}

    请严格按照上述格式和规则提取信息并输出JSON。如果用户未提及爱好，则不要在输出中包含hobby字段。\
""")

# 调用大模型API进行信息提取
completion = client.chat.completions.create(
    model="qwen3.6-flash",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "大家好，我叫刘五，今年34岁，邮箱是liuwu@example.com，平时喜欢打篮球和旅游", 
        },
    ],
    response_format={"type": "json_object"},  # 指定返回JSON格式
)

# 提取并打印模型生成的JSON结果
json_string = completion.choices[0].message.content
print(json_string)

{"info": {"name": "刘五", "age": "34岁", "email": "liuwu@example.com"}, "hobby": ["打篮球", "旅游"]}
